# Neo4j Graph Queries — Databricks Integration

Populate and query SLED graph data in Neo4j from Databricks.

**6 SLED graph models:**

| Use Case | Nodes | Key Relationships |
|---|---|---|
| `student_enrollment` | Student, Course, Department, DegreeProgram | ENROLLED_IN, OFFERED_BY, REQUIRES, PURSUING |
| `grant_budget` | FundingSource, Agency, Program, Vendor, LineItem | FUNDS, MANAGES, AWARDS, SUBCONTRACTS |
| `citizen_services` | Citizen, ServiceRequest, ServiceDepartment, Asset, District | SUBMITTED, ASSIGNED_TO, AFFECTS, IN_DISTRICT |
| `k12_early_warning` | K12Student, School, Teacher, Intervention, RiskIndicator | ATTENDS, TAUGHT_BY, RECEIVED, TARGETS |
| `procurement` | ProcAgency, ProcVendor, Contract, Lobbyist | ISSUED, AWARDED_TO, SUBCONTRACTS_TO, REPRESENTS |
| `case_management` | Client, Case, Caseworker, HHSAgency, HHSProgram | HAS_CASE, MANAGED_BY, WORKS_AT, ENROLLED_IN |

## Prerequisites
- Data API Collector stack running
- Network access from Databricks to your host (port 7688 for Neo4j via nginx/TLS)
- Databricks secrets configured

In [ ]:
%pip install neo4j
dbutils.library.restartPython()

In [ ]:
%run "./_config"


In [ ]:
import neo4j

driver = neo4j.GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_cypher(query, **kwargs):
    """Execute a Cypher query and return results as a list of dicts."""
    with driver.session() as session:
        result = session.run(query, **kwargs)
        return [record.data() for record in result]

print(run_cypher("RETURN 1 AS test"))


---
## Populate Neo4j with SLED Data

Generate graph data for all 6 use cases. Runs in background — check status after a moment.

In [ ]:
for use_case in SLED_USE_CASES:
    resp = requests.post(
        f"{API_URL}/data-sources/neo4j/sled/{use_case}/populate",
        headers=HEADERS,
        json={"num_records": 5000},
    )
    data = resp.json()
    print(f"{use_case}: job={data.get('job_id')} status={data.get('status')}")

In [ ]:
# Check populate status
import time
time.sleep(15)

for use_case in SLED_USE_CASES:
    status = requests.get(f"{API_URL}/data-sources/neo4j/sled/{use_case}/status", headers=HEADERS).json()
    total = builtins.sum(v for k, v in status["counts"].items() if k != "_relationships" and isinstance(v, int))
    print(f"{use_case}: {total} nodes")

---
## Overview — All Labels & Relationships

In [ ]:
# Node counts by label
label_counts = run_cypher("""
    MATCH (n)
    RETURN labels(n)[0] AS label, count(n) AS count
    ORDER BY count DESC
""")
display(spark.createDataFrame(label_counts))

In [ ]:
# Relationship counts by type
rel_counts = run_cypher("""
    MATCH ()-[r]->()
    RETURN type(r) AS relationship_type, count(r) AS count
    ORDER BY count DESC
""")
display(spark.createDataFrame(rel_counts))

---
## Student Enrollment — Degree Pathway Analysis

In [ ]:
# Which departments have the most enrolled students?
dept_enrollment = run_cypher("""
    MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)-[:OFFERED_BY]->(d:Department)
    RETURN d.name AS department, count(DISTINCT s) AS students, count(c) AS enrollments
    ORDER BY students DESC
""")
display(spark.createDataFrame(dept_enrollment))

In [ ]:
# Course prerequisite chains — deepest prereq trees
prereq_depth = run_cypher("""
    MATCH path = (c:Course)-[:REQUIRES*1..5]->(prereq:Course)
    RETURN c.name AS course, c.level AS level,
           length(path) AS prereq_depth,
           [n IN nodes(path) | n.name] AS chain
    ORDER BY prereq_depth DESC
    LIMIT 20
""")
display(spark.createDataFrame(prereq_depth))

In [ ]:
# Students by degree program and campus
program_dist = run_cypher("""
    MATCH (s:Student)-[:PURSUING]->(dp:DegreeProgram)-[:OFFERED_BY]->(d:Department)
    RETURN dp.degree_type AS degree, d.name AS department,
           count(s) AS students, round(avg(s.gpa), 2) AS avg_gpa
    ORDER BY students DESC
    LIMIT 20
""")
display(spark.createDataFrame(program_dist))

---
## Grant & Budget — Fund Flow Analysis

In [ ]:
# Trace funding: source -> agency -> program -> vendor
fund_flows = run_cypher("""
    MATCH (f:FundingSource)-[funds:FUNDS]->(a:Agency)-[:MANAGES]->(p:Program)-[:AWARDS]->(v:Vendor)
    RETURN f.name AS fund_source, f.category AS category,
           a.name AS agency, p.name AS program,
           v.name AS vendor, funds.amount AS amount
    ORDER BY funds.amount DESC
    LIMIT 25
""")
display(spark.createDataFrame(fund_flows))

In [ ]:
# Vendor subcontracting networks
subcontract_network = run_cypher("""
    MATCH (v1:Vendor)-[s:SUBCONTRACTS]->(v2:Vendor)
    RETURN v1.name AS prime_vendor, v2.name AS subcontractor,
           s.percentage AS subcontract_pct,
           v1.state AS prime_state, v2.state AS sub_state
    ORDER BY s.percentage DESC
    LIMIT 25
""")
display(spark.createDataFrame(subcontract_network))

---
## Citizen Services (311) — Service Request Analysis

In [ ]:
# Requests by priority
district_requests = run_cypher("""
    MATCH (sr:ServiceRequest)<-[:SUBMITTED]-(c:Citizen)
    RETURN sr.priority AS priority, count(sr) AS requests,
           collect(DISTINCT sr.type)[..5] AS sample_types
    ORDER BY requests DESC
""")
display(spark.createDataFrame(district_requests))

In [ ]:
# Assets in poor/critical condition with open service requests
at_risk_assets = run_cypher("""
    MATCH (sr:ServiceRequest)-[:AFFECTS]->(a:Asset)-[:IN_DISTRICT]->(d:District)
    WHERE a.condition IN ['poor', 'critical']
    RETURN a.asset_id AS asset, a.type AS asset_type, a.condition AS condition,
           d.name AS district, count(sr) AS open_requests
    ORDER BY open_requests DESC
    LIMIT 20
""")
display(spark.createDataFrame(at_risk_assets))

---
## K-12 Early Warning — At-Risk Student Identification

In [ ]:
# High-risk students with interventions
at_risk = run_cypher("""
    MATCH (s:K12Student)-[:ATTENDS]->(sc:School)
    WHERE s.risk_score > 70
    OPTIONAL MATCH (s)-[:RECEIVED]->(i:Intervention)
    RETURN s.student_id AS student, s.grade_level AS grade,
           s.risk_score AS risk_score, s.attendance_rate AS attendance,
           s.gpa AS gpa, sc.name AS school,
           collect(i.type) AS interventions
    ORDER BY s.risk_score DESC
    LIMIT 25
""")
display(spark.createDataFrame(at_risk))

In [ ]:
# Schools with highest average risk scores
school_risk = run_cypher("""
    MATCH (s:K12Student)-[:ATTENDS]->(sc:School)
    RETURN sc.name AS school, sc.type AS school_type,
           round(avg(s.risk_score), 1) AS avg_risk,
           round(avg(s.attendance_rate), 1) AS avg_attendance,
           count(s) AS student_count,
           sc.title_i AS title_i
    ORDER BY avg_risk DESC
    LIMIT 15
""")
display(spark.createDataFrame(school_risk))

---
## Procurement — Vendor Network Analysis

In [ ]:
# Top vendors by contract value
top_vendors = run_cypher("""
    MATCH (ag:ProcAgency)-[:ISSUED]->(ct:Contract)-[:AWARDED_TO]->(v:ProcVendor)
    RETURN v.name AS vendor, v.minority_owned AS mbe, v.small_business AS sbe,
           count(ct) AS contracts, round(sum(ct.amount)) AS total_value
    ORDER BY total_value DESC
    LIMIT 20
""")
display(spark.createDataFrame(top_vendors))

In [ ]:
# Lobbyist -> Vendor -> Contract connections
lobby_contracts = run_cypher("""
    MATCH (lb:Lobbyist)-[:REPRESENTS]->(v:ProcVendor)<-[:AWARDED_TO]-(ct:Contract)<-[:ISSUED]-(ag:ProcAgency)
    RETURN lb.name AS lobbyist, lb.firm AS firm,
           v.name AS vendor, ag.name AS agency,
           ct.amount AS contract_value, ct.method AS procurement_method
    ORDER BY ct.amount DESC
    LIMIT 20
""")
display(spark.createDataFrame(lobby_contracts))

---
## Case Management (HHS) — Client Service Analysis

In [ ]:
# Clients enrolled in multiple programs
multi_program = run_cypher("""
    MATCH (cl:Client)-[:HAS_CASE]->(cs:Case)-[:ENROLLED_IN]->(pg:HHSProgram)
    WITH cl, collect(DISTINCT pg.name) AS programs, count(DISTINCT cs) AS cases
    WHERE size(programs) > 1
    RETURN cl.client_id AS client, cl.income_bracket AS income,
           cl.household_size AS household_size,
           cases, programs
    ORDER BY size(programs) DESC
    LIMIT 20
""")
display(spark.createDataFrame(multi_program))

In [ ]:
# Caseworker caseload analysis
caseload = run_cypher("""
    MATCH (cs:Case)-[:MANAGED_BY]->(cw:Caseworker)-[:WORKS_AT]->(ag:HHSAgency)
    RETURN cw.name AS caseworker, cw.specialization AS specialization,
           ag.name AS agency, count(cs) AS active_cases,
           round(avg(cs.benefit_amount), 2) AS avg_benefit
    ORDER BY active_cases DESC
    LIMIT 20
""")
display(spark.createDataFrame(caseload))

---
## Write Neo4j Data to Delta Tables

Export graph query results to Delta tables in Unity Catalog for downstream analytics.


In [ ]:
# Helper: run a Cypher query and save results as a Delta table
def cypher_to_delta(query, table_name, mode="overwrite"):
    """Run a Cypher query, convert to Spark DataFrame, and write to Delta."""
    results = run_cypher(query)
    if not results:
        print(f"  {table_name}: no results, skipping")
        return None
    df = spark.createDataFrame(results)
    df.write.format("delta").mode(mode).saveAsTable(f"{DELTA_BASE}.{table_name}")
    print(f"  {table_name}: {df.count()} rows written")
    return df


In [ ]:
# Student Enrollment
cypher_to_delta("""
    MATCH (s:Student)-[:ENROLLED_IN]->(c:Course)-[:OFFERED_BY]->(d:Department)
    RETURN s.student_id AS student_id, s.name AS student_name, s.gpa AS gpa,
           s.campus AS campus, s.status AS status,
           c.course_id AS course_id, c.name AS course_name, c.level AS course_level, c.credits AS credits,
           d.name AS department, d.college AS college
""", "neo4j_student_enrollment")

# Student degree programs
cypher_to_delta("""
    MATCH (s:Student)-[p:PURSUING]->(dp:DegreeProgram)-[:OFFERED_BY]->(d:Department)
    RETURN s.student_id AS student_id, s.name AS student_name,
           dp.program_id AS program_id, dp.degree_type AS degree_type, dp.name AS program_name,
           d.name AS department, p.start_semester AS start_semester, p.expected_graduation AS expected_graduation
""", "neo4j_degree_pathways")


In [ ]:
# Grant & Budget — fund flows
cypher_to_delta("""
    MATCH (f:FundingSource)-[funds:FUNDS]->(a:Agency)-[:MANAGES]->(p:Program)
    RETURN f.fund_id AS fund_id, f.name AS fund_name, f.category AS fund_category, f.total_budget AS total_budget,
           a.agency_id AS agency_id, a.name AS agency_name, a.type AS agency_type,
           p.program_id AS program_id, p.name AS program_name, p.budget_allocated AS program_budget,
           funds.amount AS funded_amount
""", "neo4j_fund_flows")

# Vendor subcontracting network
cypher_to_delta("""
    MATCH (v1:Vendor)-[s:SUBCONTRACTS]->(v2:Vendor)
    RETURN v1.vendor_id AS prime_vendor_id, v1.name AS prime_vendor, v1.state AS prime_state,
           v2.vendor_id AS sub_vendor_id, v2.name AS subcontractor, v2.state AS sub_state,
           s.percentage AS subcontract_pct
""", "neo4j_vendor_subcontracts")


In [ ]:
# Citizen Services — requests with assets and districts
cypher_to_delta("""
    MATCH (cit:Citizen)-[:SUBMITTED]->(sr:ServiceRequest)
    OPTIONAL MATCH (sr)-[:ASSIGNED_TO]->(dept:ServiceDepartment)
    OPTIONAL MATCH (sr)-[:AFFECTS]->(a:Asset)-[:IN_DISTRICT]->(d:District)
    RETURN sr.request_id AS request_id, sr.type AS request_type, sr.status AS status, sr.priority AS priority,
           cit.citizen_id AS citizen_id, cit.name AS citizen_name,
           dept.name AS department,
           a.asset_id AS asset_id, a.type AS asset_type, a.condition AS asset_condition,
           d.name AS district
    LIMIT 50000
""", "neo4j_citizen_services")


In [ ]:
# K-12 — at-risk students with school and interventions
cypher_to_delta("""
    MATCH (s:K12Student)-[:ATTENDS]->(sc:School)
    OPTIONAL MATCH (s)-[:RECEIVED]->(i:Intervention)
    RETURN s.student_id AS student_id, s.name AS student_name, s.grade_level AS grade_level,
           s.risk_score AS risk_score, s.attendance_rate AS attendance_rate, s.gpa AS gpa,
           s.free_reduced_lunch AS frl, s.english_learner AS ell, s.special_education AS sped,
           sc.school_id AS school_id, sc.name AS school_name, sc.type AS school_type, sc.title_i AS title_i,
           i.intervention_id AS intervention_id, i.type AS intervention_type, i.status AS intervention_status,
           i.effectiveness AS effectiveness
""", "neo4j_k12_early_warning")


In [ ]:
# Procurement — contracts with vendors and lobbyists
cypher_to_delta("""
    MATCH (ag:ProcAgency)-[:ISSUED]->(ct:Contract)-[:AWARDED_TO]->(v:ProcVendor)
    OPTIONAL MATCH (lb:Lobbyist)-[:REPRESENTS]->(v)
    RETURN ag.agency_id AS agency_id, ag.name AS agency_name,
           ct.contract_id AS contract_id, ct.title AS contract_title, ct.amount AS amount,
           ct.method AS procurement_method, ct.category AS category, ct.status AS contract_status,
           v.vendor_id AS vendor_id, v.name AS vendor_name,
           v.minority_owned AS mbe, v.small_business AS sbe, v.woman_owned AS wbe,
           lb.name AS lobbyist_name, lb.firm AS lobbyist_firm
""", "neo4j_procurement")


In [ ]:
# Case Management — clients, cases, programs
cypher_to_delta("""
    MATCH (cl:Client)-[:HAS_CASE]->(cs:Case)-[:MANAGED_BY]->(cw:Caseworker)-[:WORKS_AT]->(ag:HHSAgency)
    OPTIONAL MATCH (cs)-[:ENROLLED_IN]->(pg:HHSProgram)
    RETURN cl.client_id AS client_id, cl.name AS client_name,
           cl.income_bracket AS income_bracket, cl.household_size AS household_size,
           cs.case_id AS case_id, cs.status AS case_status, cs.priority AS priority,
           cs.determination AS determination, cs.benefit_amount AS benefit_amount,
           cw.caseworker_id AS caseworker_id, cw.name AS caseworker_name, cw.specialization AS specialization,
           ag.name AS agency_name,
           pg.name AS program_name
""", "neo4j_case_management")


In [ ]:
print("\nAll Neo4j Delta tables written. Summary:")
for table in ["neo4j_student_enrollment", "neo4j_degree_pathways", "neo4j_fund_flows",
              "neo4j_vendor_subcontracts", "neo4j_citizen_services", "neo4j_k12_early_warning",
              "neo4j_procurement", "neo4j_case_management"]:
    try:
        count = spark.table(f"{DELTA_BASE}.{table}").count()
        print(f"  {DELTA_BASE}.{table}: {count:,} rows")
    except Exception:
        print(f"  {DELTA_BASE}.{table}: not found")


---
## Cleanup

In [ ]:
driver.close()
print("Neo4j driver closed")

# Uncomment to clear all SLED graph data
# for use_case in SLED_USE_CASES:
#     resp = requests.delete(f"{API_URL}/data-sources/neo4j/sled/{use_case}/clear", headers=HEADERS)
#     print(f"{use_case}: {resp.json()}")